In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents_raw = []

for file in files:
    doc = file.parse()
    documents_raw.append(doc)

In [3]:
documents_raw[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [4]:
len(documents_raw)

72

In [5]:
from gitsource import chunk_documents

documents = chunk_documents(documents_raw, size=2000, step=1000)

In [6]:
from minsearch import Index

In [7]:
def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

In [8]:
index = build_index(documents)

In [9]:
query = "How does the agentic loop keep calling the model until it stops?"

In [10]:
search_result = index.search(
            query,
            num_results=5,
        )

In [29]:
search_result

[{'start': 4000,
  'content': 'while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\niteration counter so we can see how many round-trips happened.\n\n```python\nit = 1\n\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n\n        elif item.type == "message":\n            print("ASSISTANT:")\n            print(item.content[0].text)\n\n    it = it + 1\n    if has_function_calls == False:\n        break\n```\n\nThis is the core agent loop. The model rea

In [30]:
search_result[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [31]:
from rag_helper import RAGNew

In [11]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

openai_client = OpenAI()

In [33]:
assistant = RAGNew(
    index=index,
    llm_client=openai_client,
)

In [34]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [35]:
print(answer)

Response(id='resp_02c5f706b894383a006a31c72f9c04819dac4ea97ff1934596', created_at=1781647151.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_02c5f706b894383a006a31c72fe4e8819da6659a8608da2065', content=[ResponseOutputText(annotations=[], text='It keeps calling the model inside a `while True` loop. After each model response, it checks whether any item is a `function_call`:\n\n- If there is a function call, it runs the tool, adds the result to `messages`, and keeps looping.\n- If there are no function calls in that turn, it breaks out of the loop and stops.\n\nSo the stop condition is: **no function calls returned by the model in that iteration**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1

In [36]:
print(answer.output_text)

It keeps calling the model inside a `while True` loop. After each model response, it checks whether any item is a `function_call`:

- If there is a function call, it runs the tool, adds the result to `messages`, and keeps looping.
- If there are no function calls in that turn, it breaks out of the loop and stops.

So the stop condition is: **no function calls returned by the model in that iteration**.


In [37]:
print(answer.usage)

ResponseUsage(input_tokens=2295, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2392)


In [12]:
def search(query, num_results=5):
        boost_dict = {"content": 3.0, "filename": 0.5}

        return index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
        )

In [13]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the lessons for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the content ofcourse lessons."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [19]:
import json

In [20]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [21]:
question = "How does the agentic loop work, and how is it different from plain RAG?"


In [22]:
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [23]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [24]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)
    print(response.output)
    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
[ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG"}', call_id='call_UwuhGsiUrCbQPfVoBjreXLIV', name='search', type='function_call', id='fc_099a4f187e759b49006a31cb48e9448191be386eb9e3b66713', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"plain RAG"}', call_id='call_1sk29lu2xQyGlxe7ynsWlgx0', name='search', type='function_call', id='fc_099a4f187e759b49006a31cb48e9588191989fe13e38845056', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop"}', call_id='call_DYjngBze2TB2Hvv265j7BCU9', name='search', type='function_call', id='fc_099a4f187e759b49006a31cb48e968819195921c761b8a6d2d', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"RAG retrieval generation"}', call_id='call_unrJltUWVXSGiUHJssG52O2h', name='search', type='function_call', id='fc_099a4f187e759b49006a31cb48e9788191a948114cf3930f38', namespace=None, status='completed')]
function_call: sea